# Day 013 | Decorators & Generators

## 100 Days of ML for Geotechnical Engineering
**Phase:** Python Fundamentals

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| First-class functions | Functions as values you can pass around | Pass different soil-property formulas to a generic calculator |
| Closures | A function that "remembers" its creation environment | Factory that builds SPT correction functions for each hammer type |
| `@decorator` syntax | Wrap a function to add behavior | `@timer` to measure how long a settlement calculation takes |
| `@validate_input` | Check inputs before running | Ensure N-values are positive before computing |
| `@log_calculation` | Record every engineering calculation | Automatic audit trail for bearing capacity computations |
| `yield` generators | Produce values one at a time, on demand | Stream soil layers from a borehole log without loading all into memory |
| Generator expressions | One-line lazy sequences | `(layer for layer in borehole if layer["N60"] > 15)` |
| `itertools` | Efficient iterator combinators | `chain` boreholes together, `islice` first N layers |

---

## Our Geotechnical Problem

As geotechnical projects grow, we face two recurring challenges:

**Challenge 1 -- Repetitive boilerplate code.** Every engineering function needs input validation, timing, and logging. We end up copy-pasting the same checks into dozens of functions:

```python
def bearing_capacity(N60, depth):
    if N60 <= 0: raise ValueError("N must be positive")   # repeated
    start = time.time()                                     # repeated
    result = ...                                            # actual work
    print(f"Took {time.time()-start:.3f}s")                # repeated
    return result
```

**Decorators** let us write validation, timing, and logging **once** and apply them to any function with a single line.

**Challenge 2 -- Large datasets.** A site investigation with 50 boreholes, each with 20 layers, means 1000 layer records. Loading everything into a list wastes memory when we only process one layer at a time.

**Generators** produce data on demand -- one layer at a time -- keeping memory usage constant no matter how large the dataset.

---

## Part 1: Functions as First-Class Objects

In Python, functions are **objects** -- you can assign them to variables, store them in lists, and pass them as arguments to other functions. This is the foundation that makes decorators possible.

```python
# A function IS a value:
my_func = some_function    # no parentheses = reference to the function
my_func(args)              # call it through the new name
```

This lets us build **generic calculators** that accept different soil-property formulas as arguments.

In [1]:
# ─────────────────────────────────────────────────────
# FUNCTIONS AS FIRST-CLASS OBJECTS
# Passing soil-property formulas around like data
# ─────────────────────────────────────────────────────

import math

# Two different formulas for shear wave velocity from SPT N-value
def vs_imai_1977(n60):
    """Vs = 97 * N^0.314  (Imai & Tonouchi, 1982)"""
    return 97.0 * (n60 ** 0.314)

def vs_ohta_goto_1978(n60):
    """Vs = 85.35 * N^0.348  (Ohta & Goto, 1978)"""
    return 85.35 * (n60 ** 0.348)

# Functions are just objects -- assign them to variables
formula = vs_imai_1977
print(f"formula is: {formula.__name__}")
print(f"Vs(20) = {formula(20):.1f} m/s")
print()

# Store functions in a list
formulas = [vs_imai_1977, vs_ohta_goto_1978]
print("Comparing Vs correlations for N60=20:")
for f in formulas:
    print(f"  {f.__name__:20s} -> Vs = {f(20):.1f} m/s")
print()

# Pass a function as an argument to another function
def apply_to_borehole(n_values, formula_func):
    """Apply any Vs formula to an entire borehole profile."""
    return [round(formula_func(n), 1) for n in n_values]

n_values = [5, 8, 12, 18, 28, 35, 50]
print(f"N-values: {n_values}")
print(f"Imai Vs:      {apply_to_borehole(n_values, vs_imai_1977)}")
print(f"Ohta-Goto Vs: {apply_to_borehole(n_values, vs_ohta_goto_1978)}")

formula is: vs_imai_1977
Vs(20) = 248.5 m/s

Comparing Vs correlations for N60=20:
  vs_imai_1977         -> Vs = 248.5 m/s
  vs_ohta_goto_1978    -> Vs = 242.1 m/s

N-values: [5, 8, 12, 18, 28, 35, 50]
Imai Vs:      [160.8, 186.4, 211.7, 240.4, 276.2, 296.2, 331.3]
Ohta-Goto Vs: [149.4, 176.0, 202.7, 233.4, 272.2, 294.1, 333.0]


---

## Part 2: Closures -- Function Factories

A **closure** is a function defined inside another function that "remembers" the variables from its enclosing scope, even after the outer function has finished.

This lets us build **function factories** -- functions that create and return customized functions.

**Geotechnical use case:** SPT energy correction. Different hammer types deliver different energy ratios (ER). We can build a factory that produces a correction function for each hammer type:

```
N60 = N_field * (ER / 60)
```

| Hammer Type | Energy Ratio (ER) |
|-------------|-------------------|
| Donut       | 45% |
| Safety      | 60% |
| Automatic   | 80% |
| Pin-weight  | 50% |

In [2]:
# ─────────────────────────────────────────────────────
# CLOSURES -- SPT hammer correction function factory
# ─────────────────────────────────────────────────────

def make_spt_corrector(hammer_type, energy_ratio):
    """
    Factory: returns a function that corrects raw N to N60
    for a specific hammer type.
    
    The returned function 'remembers' hammer_type and energy_ratio
    even after make_spt_corrector() has finished -- that is a closure.
    """
    def correct(n_field):
        n60 = n_field * (energy_ratio / 60.0)
        return round(n60, 1)
    
    # Attach metadata for clarity
    correct.__doc__ = f"Correct N to N60 for {hammer_type} hammer (ER={energy_ratio}%)"
    correct.hammer = hammer_type
    correct.er = energy_ratio
    return correct

# Create specialized corrector functions
correct_donut = make_spt_corrector("Donut", 45)
correct_safety = make_spt_corrector("Safety", 60)
correct_auto = make_spt_corrector("Automatic", 80)

# Each function 'remembers' its hammer type
n_field = 25
print(f"Raw field N = {n_field}")
print(f"  Donut hammer (ER=45%):     N60 = {correct_donut(n_field)}")
print(f"  Safety hammer (ER=60%):    N60 = {correct_safety(n_field)}")
print(f"  Automatic hammer (ER=80%): N60 = {correct_auto(n_field)}")
print()

# Apply to an entire borehole profile
n_field_values = [5, 10, 15, 22, 30, 40, 50]
print(f"{'N_field':>7s} {'Donut':>7s} {'Safety':>7s} {'Auto':>7s}")
print("─" * 32)
for n in n_field_values:
    print(f"{n:7d} {correct_donut(n):7.1f} {correct_safety(n):7.1f} {correct_auto(n):7.1f}")

print()
print(f"correct_donut remembers: hammer={correct_donut.hammer}, ER={correct_donut.er}%")
print("The closure 'captured' those values when it was created.")

Raw field N = 25
  Donut hammer (ER=45%):     N60 = 18.8
  Safety hammer (ER=60%):    N60 = 25.0
  Automatic hammer (ER=80%): N60 = 33.3

N_field   Donut  Safety    Auto
────────────────────────────────
      5     3.8     5.0     6.7
     10     7.5    10.0    13.3
     15    11.2    15.0    20.0
     22    16.5    22.0    29.3
     30    22.5    30.0    40.0
     40    30.0    40.0    53.3
     50    37.5    50.0    66.7

correct_donut remembers: hammer=Donut, ER=45%
The closure 'captured' those values when it was created.


---

## Part 3: Decorators Basics -- The `@decorator` Syntax

A **decorator** is a function that takes a function as input and returns a new, enhanced function. The `@` syntax is just shorthand:

```python
# These two are identical:

@my_decorator                    # Shorthand
def my_func(): ...               

def my_func(): ...               # Longhand
my_func = my_decorator(my_func)
```

The pattern:
1. Define a decorator function that accepts `func`
2. Inside, define a `wrapper` that calls `func` with extra behavior
3. Return the `wrapper`

In [3]:
# ─────────────────────────────────────────────────────
# DECORATOR BASICS -- Understanding the pattern
# ─────────────────────────────────────────────────────

import functools

def simple_logger(func):
    """A basic decorator that prints when a function is called."""
    @functools.wraps(func)  # preserves original function's name & docstring
    def wrapper(*args, **kwargs):
        print(f"  [LOG] Calling {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"  [LOG] {func.__name__} returned {result}")
        return result
    return wrapper

# Apply decorator with @ syntax
@simple_logger
def compute_overburden(gamma, depth):
    """sigma_v = gamma * depth"""
    return round(gamma * depth, 2)

@simple_logger
def compute_vs(n60):
    """Vs from Imai correlation."""
    return round(97.0 * (n60 ** 0.314), 1)

# Now every call is automatically logged
print("Overburden stress:")
sigma = compute_overburden(18.5, 6.0)
print()

print("Shear wave velocity:")
vs = compute_vs(25)
print()

# The function still has its original name and docstring
print(f"Function name: {compute_overburden.__name__}")
print(f"Docstring:     {compute_overburden.__doc__}")

Overburden stress:
  [LOG] Calling compute_overburden...
  [LOG] compute_overburden returned 111.0

Shear wave velocity:
  [LOG] Calling compute_vs...
  [LOG] compute_vs returned 266.5

Function name: compute_overburden
Docstring:     sigma_v = gamma * depth


---

## Part 4: Practical Decorator -- `@timer`

Measure how long any geotechnical computation takes. This is critical when processing large site investigation datasets -- you need to know which functions are bottlenecks.

In [4]:
# ─────────────────────────────────────────────────────
# @timer -- Measure computation time
# ─────────────────────────────────────────────────────

import time
import functools

def timer(func):
    """Decorator: prints execution time of the wrapped function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  [TIMER] {func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def settlement_schmertmann(q_net, layer_thicknesses, elastic_moduli, iz_values):
    """
    Schmertmann (1970) settlement calculation.
    S = C1 * C2 * q_net * sum(Iz * dz / Es)
    """
    total = 0.0
    for dz, Es, Iz in zip(layer_thicknesses, elastic_moduli, iz_values):
        total += Iz * dz / Es
    settlement = q_net * total * 1000  # convert to mm
    return round(settlement, 2)

@timer
def compute_vs_profile(n_values, depths):
    """Compute Vs for an entire borehole profile."""
    profile = []
    for n, d in zip(n_values, depths):
        vs = 97.0 * (n ** 0.314)
        profile.append({"depth": d, "N60": n, "Vs": round(vs, 1)})
    return profile

# Test the timer
print("Settlement calculation:")
s = settlement_schmertmann(
    q_net=120.0,
    layer_thicknesses=[0.5, 0.5, 0.5, 0.5, 0.5, 0.5],
    elastic_moduli=[8000, 12000, 15000, 18000, 22000, 25000],
    iz_values=[0.2, 0.4, 0.55, 0.5, 0.35, 0.15]
)
print(f"  Settlement = {s} mm\n")

print("Vs profile computation:")
profile = compute_vs_profile(
    [5, 8, 12, 18, 28, 35, 50, 60],
    [1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0]
)
for p in profile:
    print(f"    {p['depth']:5.1f}m  N={p['N60']:3d}  Vs={p['Vs']:6.1f} m/s")

Settlement calculation:
  [TIMER] settlement_schmertmann took 0.000009s
  Settlement = 8.68 mm

Vs profile computation:
  [TIMER] compute_vs_profile took 0.000012s
      1.5m  N=  5  Vs= 160.8 m/s
      3.0m  N=  8  Vs= 186.4 m/s
      4.5m  N= 12  Vs= 211.7 m/s
      6.0m  N= 18  Vs= 240.4 m/s
      7.5m  N= 28  Vs= 276.2 m/s
      9.0m  N= 35  Vs= 296.2 m/s
     10.5m  N= 50  Vs= 331.3 m/s
     12.0m  N= 60  Vs= 350.8 m/s


---

## Part 5: Practical Decorator -- `@validate_input`

In geotechnical engineering, garbage in = garbage out. N-values must be positive, depths must be non-negative, unit weights must be realistic. Instead of repeating validation code in every function, we write it **once** as a decorator.

In [5]:
# ─────────────────────────────────────────────────────
# @validate_input -- Check engineering inputs
# ─────────────────────────────────────────────────────

import functools

def validate_positive(*param_names):
    """
    Decorator factory: ensures named parameters are positive.
    Usage: @validate_positive('n60', 'depth')
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Get parameter names from the function signature
            import inspect
            sig = inspect.signature(func)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            
            # Check each specified parameter
            for name in param_names:
                value = bound.arguments.get(name)
                if value is not None and value <= 0:
                    raise ValueError(
                        f"{func.__name__}: '{name}' must be positive, got {value}"
                    )
            return func(*args, **kwargs)
        return wrapper
    return decorator

@validate_positive('n60', 'depth')
def bearing_capacity_meyerhof(n60, depth, width=1.0):
    """
    Meyerhof (1965) allowable bearing capacity for sand.
    q_a = N60 * Kd / 12  (in ksf, simplified)
    """
    kd = 1.0 + 0.33 * (depth / width)
    if kd > 1.33:
        kd = 1.33
    q_a = n60 * kd / 12.0
    return round(q_a, 2)

# Valid call
print("Valid inputs:")
q = bearing_capacity_meyerhof(n60=20, depth=3.0, width=2.0)
print(f"  q_a = {q} ksf")
print()

# Invalid call -- caught by decorator
print("Invalid inputs (negative N60):")
try:
    bearing_capacity_meyerhof(n60=-5, depth=3.0)
except ValueError as e:
    print(f"  Caught: {e}")

print()
print("Invalid inputs (zero depth):")
try:
    bearing_capacity_meyerhof(n60=20, depth=0)
except ValueError as e:
    print(f"  Caught: {e}")

print()
print("The validation logic is written ONCE and reusable on any function.")

Valid inputs:
  q_a = 2.22 ksf

Invalid inputs (negative N60):
  Caught: bearing_capacity_meyerhof: 'n60' must be positive, got -5

Invalid inputs (zero depth):
  Caught: bearing_capacity_meyerhof: 'depth' must be positive, got 0

The validation logic is written ONCE and reusable on any function.


---

## Part 6: Practical Decorator -- `@log_calculation`

In professional engineering, every calculation needs an **audit trail**. The `@log_calculation` decorator automatically records every call with inputs, outputs, and timestamps -- essential for design verification.

In [6]:
# ─────────────────────────────────────────────────────
# @log_calculation -- Automatic engineering audit trail
# ─────────────────────────────────────────────────────

import functools
from datetime import datetime

# Global calculation log
calculation_log = []

def log_calculation(func):
    """Decorator: records every call to a calculation log."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "function": func.__name__,
            "args": args,
            "kwargs": kwargs,
            "result": result,
        }
        calculation_log.append(entry)
        return result
    return wrapper

@log_calculation
def overburden_stress(gamma, depth):
    """sigma_v = gamma * depth  (kPa)"""
    return round(gamma * depth, 2)

@log_calculation
def effective_stress(sigma_v, u):
    """sigma_v_prime = sigma_v - u  (kPa)"""
    return round(sigma_v - u, 2)

@log_calculation
def n_correction_overburden(n60, sigma_v_eff):
    """Liao & Whitman (1986): (N1)60 = N60 * sqrt(100/sigma_v')"""
    if sigma_v_eff <= 0:
        return n60
    cn = min(math.sqrt(100.0 / sigma_v_eff), 2.0)
    return round(n60 * cn, 1)

# Run a series of calculations
print("Running SPT correction calculations...\n")

gamma = 18.5  # kN/m3
depth = 6.0   # m
wt = 3.0      # m
gamma_w = 9.81
n60 = 20

sv = overburden_stress(gamma, depth)
u = overburden_stress(gamma_w, depth - wt)
sv_eff = effective_stress(sv, u)
n1_60 = n_correction_overburden(n60, sv_eff)

print(f"  sigma_v  = {sv} kPa")
print(f"  u        = {u} kPa")
print(f"  sigma_v' = {sv_eff} kPa")
print(f"  (N1)60   = {n1_60}")
print()

# Print the audit trail
print("CALCULATION AUDIT LOG:")
print("─" * 70)
for entry in calculation_log:
    args_str = ", ".join(str(a) for a in entry["args"])
    print(f"  {entry['timestamp']} | {entry['function']:28s} "
          f"| args=({args_str}) -> {entry['result']}")
print("─" * 70)
print(f"Total calculations logged: {len(calculation_log)}")

Running SPT correction calculations...

  sigma_v  = 111.0 kPa
  u        = 29.43 kPa
  sigma_v' = 81.57 kPa
  (N1)60   = 22.1

CALCULATION AUDIT LOG:
──────────────────────────────────────────────────────────────────────
  2026-11-15 15:06:21 | overburden_stress            | args=(18.5, 6.0) -> 111.0
  2026-11-15 15:06:21 | overburden_stress            | args=(9.81, 3.0) -> 29.43
  2026-11-15 15:06:21 | effective_stress             | args=(111.0, 29.43) -> 81.57
  2026-11-15 15:06:21 | n_correction_overburden      | args=(20, 81.57) -> 22.1
──────────────────────────────────────────────────────────────────────
Total calculations logged: 4


---

## Part 7: Stacking Decorators

You can apply **multiple decorators** to a single function. They execute from bottom to top (closest to the function first):

```python
@timer               # 3rd: outermost, measures total time
@log_calculation     # 2nd: logs the call
@validate_positive('n60')  # 1st: validates inputs
def my_func(n60): ...
```

In [7]:
# ─────────────────────────────────────────────────────
# STACKING DECORATORS -- Combine multiple behaviors
# ─────────────────────────────────────────────────────

# Reset log for clarity
calculation_log.clear()

@timer
@log_calculation
@validate_positive('n60', 'depth')
def corrected_bearing_capacity(n60, depth, width=1.5, gamma=18.0):
    """
    Full bearing capacity with corrections.
    Applies overburden correction, then Meyerhof formula.
    """
    # Overburden correction
    sigma_v = gamma * depth
    cn = min(math.sqrt(100.0 / sigma_v), 2.0) if sigma_v > 0 else 1.0
    n1_60 = n60 * cn
    
    # Meyerhof bearing capacity
    kd = min(1.0 + 0.33 * (depth / width), 1.33)
    q_a = n1_60 * kd / 12.0
    
    return round(q_a, 2)

# Call with valid inputs -- all three decorators fire
print("Valid call (stacked decorators):")
q = corrected_bearing_capacity(n60=25, depth=4.0, width=2.0)
print(f"  q_a = {q} ksf\n")

# Call with invalid input -- validation catches it before logging or timing
print("Invalid call:")
try:
    corrected_bearing_capacity(n60=-10, depth=4.0)
except ValueError as e:
    print(f"  Caught: {e}")

print(f"\nLog entries: {len(calculation_log)} (invalid call was blocked)")

Valid call (stacked decorators):
  [TIMER] corrected_bearing_capacity took 0.000097s
  q_a = 3.27 ksf

Invalid call:
  Caught: corrected_bearing_capacity: 'n60' must be positive, got -10

Log entries: 1 (invalid call was blocked)


---

## Part 8: Generators with `yield` -- Streaming Borehole Data

A **generator** is a function that uses `yield` instead of `return`. Each time you call `next()` on it (or loop over it), it runs until the next `yield`, produces a value, then **pauses**.

```python
# Regular function: builds entire list in memory
def get_all_layers():      
    return [layer1, layer2, ..., layer1000]  # 1000 items in RAM

# Generator: yields one layer at a time
def stream_layers():       
    yield layer1   # pause, give layer1
    yield layer2   # pause, give layer2
    ...            # only 1 item in memory at a time!
```

**Why it matters:** A real site investigation might have 50+ boreholes with 20+ layers each. Generators let you process them one at a time without loading the entire dataset.

In [8]:
# ─────────────────────────────────────────────────────
# GENERATORS -- Stream soil layers from a borehole log
# ─────────────────────────────────────────────────────

def borehole_layers(borehole_id, layers_data):
    """
    Generator: yields one soil layer at a time from a borehole log.
    Memory-efficient -- only one layer in memory at any moment.
    """
    print(f"  [GEN] Starting to read {borehole_id}...")
    for i, layer in enumerate(layers_data):
        record = {
            "borehole": borehole_id,
            "layer_num": i + 1,
            **layer
        }
        print(f"  [GEN] Yielding layer {i+1}...")
        yield record  # PAUSE here, hand off one layer
    print(f"  [GEN] {borehole_id} exhausted.")

# Borehole data
bh01_data = [
    {"depth": 1.5, "N60": 5,  "soil": "Fill"},
    {"depth": 3.0, "N60": 8,  "soil": "Silty Sand"},
    {"depth": 4.5, "N60": 15, "soil": "Sand"},
    {"depth": 6.0, "N60": 22, "soil": "Clayey Sand"},
    {"depth": 7.5, "N60": 35, "soil": "Dense Sand"},
]

# Create the generator (nothing runs yet!)
gen = borehole_layers("BH-01", bh01_data)
print(f"Generator created: {type(gen)}")
print()

# Pull values one at a time with next()
print("Pulling layers manually:")
layer1 = next(gen)
print(f"  Got: {layer1['depth']}m - {layer1['soil']}\n")

layer2 = next(gen)
print(f"  Got: {layer2['depth']}m - {layer2['soil']}\n")

# Loop through remaining layers
print("Looping through the rest:")
for layer in gen:
    print(f"  Processing: {layer['depth']}m | N={layer['N60']} | {layer['soil']}")

Generator created: <class 'generator'>

Pulling layers manually:
  [GEN] Starting to read BH-01...
  [GEN] Yielding layer 1...
  Got: 1.5m - Fill

  [GEN] Yielding layer 2...
  Got: 3.0m - Silty Sand

Looping through the rest:
  [GEN] Yielding layer 3...
  Processing: 4.5m | N=15 | Sand
  [GEN] Yielding layer 4...
  Processing: 6.0m | N=22 | Clayey Sand
  [GEN] Yielding layer 5...
  Processing: 7.5m | N=35 | Dense Sand
  [GEN] BH-01 exhausted.


In [9]:
# ─────────────────────────────────────────────────────
# GENERATOR WITH FILTERING -- yield only layers that
# meet engineering criteria
# ─────────────────────────────────────────────────────

def competent_layers(layers_data, min_n60=15):
    """
    Generator: yields only layers with N60 >= min_n60.
    Useful for finding suitable foundation bearing strata.
    """
    for layer in layers_data:
        if layer["N60"] >= min_n60:
            yield layer

def layers_between_depths(layers_data, top, bottom):
    """
    Generator: yields only layers within a depth range.
    Useful for analyzing a specific zone (e.g., foundation influence zone).
    """
    for layer in layers_data:
        if top <= layer["depth"] <= bottom:
            yield layer

# Sample data
all_layers = [
    {"depth": 1.5, "N60": 5,  "soil": "Fill"},
    {"depth": 3.0, "N60": 8,  "soil": "Silty Sand"},
    {"depth": 4.5, "N60": 15, "soil": "Sand"},
    {"depth": 6.0, "N60": 22, "soil": "Clayey Sand"},
    {"depth": 7.5, "N60": 35, "soil": "Dense Sand"},
    {"depth": 9.0, "N60": 42, "soil": "Gravel"},
    {"depth": 10.5, "N60": 55, "soil": "Weathered Rock"},
]

# Find competent bearing layers
print("Competent layers (N60 >= 15):")
for layer in competent_layers(all_layers, min_n60=15):
    print(f"  {layer['depth']:5.1f}m  N={layer['N60']:3d}  {layer['soil']}")

print()

# Layers in foundation influence zone (3m to 7.5m)
print("Layers in influence zone (3.0m - 7.5m):")
for layer in layers_between_depths(all_layers, 3.0, 7.5):
    print(f"  {layer['depth']:5.1f}m  N={layer['N60']:3d}  {layer['soil']}")

print()

# Chain generators: competent layers within influence zone
print("Competent layers IN influence zone:")
zone_layers = layers_between_depths(all_layers, 3.0, 7.5)
for layer in competent_layers(zone_layers, min_n60=15):
    print(f"  {layer['depth']:5.1f}m  N={layer['N60']:3d}  {layer['soil']}")

Competent layers (N60 >= 15):
    4.5m  N= 15  Sand
    6.0m  N= 22  Clayey Sand
    7.5m  N= 35  Dense Sand
    9.0m  N= 42  Gravel
   10.5m  N= 55  Weathered Rock

Layers in influence zone (3.0m - 7.5m):
    3.0m  N=  8  Silty Sand
    4.5m  N= 15  Sand
    6.0m  N= 22  Clayey Sand
    7.5m  N= 35  Dense Sand

Competent layers IN influence zone:
    4.5m  N= 15  Sand
    6.0m  N= 22  Clayey Sand
    7.5m  N= 35  Dense Sand


---

## Part 9: Generator Expressions vs List Comprehensions

A **generator expression** looks exactly like a list comprehension but uses `()` instead of `[]`. The difference: it produces values lazily instead of building the entire list.

| Syntax | Type | Memory | When to Use |
|--------|------|--------|-------------|
| `[x for x in data]` | `list` | Stores all values at once | Need to index, slice, or reuse |
| `(x for x in data)` | `generator` | One value at a time | Single pass, large dataset |

In [10]:
# ─────────────────────────────────────────────────────
# GENERATOR EXPRESSIONS vs LIST COMPREHENSIONS
# ─────────────────────────────────────────────────────

import sys

n_values = [5, 8, 12, 18, 22, 28, 35, 42, 50, 60]

# List comprehension -- builds entire list in memory
vs_list = [round(97.0 * (n ** 0.314), 1) for n in n_values]

# Generator expression -- computes one at a time
vs_gen = (round(97.0 * (n ** 0.314), 1) for n in n_values)

print(f"List type:      {type(vs_list)}")
print(f"Generator type: {type(vs_gen)}")
print(f"List memory:    {sys.getsizeof(vs_list)} bytes")
print(f"Generator mem:  {sys.getsizeof(vs_gen)} bytes")
print()

# With a list, you can index and reuse
print(f"vs_list[0] = {vs_list[0]}")
print(f"vs_list[3] = {vs_list[3]}")
print()

# With a generator, iterate once
print("Generator values (consumed one by one):")
for vs in vs_gen:
    print(f"  {vs}", end="")
print()
print()

# Practical: use generator directly in sum(), max(), min()
# No intermediate list needed!
avg_vs = sum(97.0 * (n ** 0.314) for n in n_values) / len(n_values)
max_vs = max(97.0 * (n ** 0.314) for n in n_values)
min_vs = min(97.0 * (n ** 0.314) for n in n_values)

print(f"Avg Vs = {avg_vs:.1f} m/s")
print(f"Max Vs = {max_vs:.1f} m/s")
print(f"Min Vs = {min_vs:.1f} m/s")
print()
print("Tip: Use generator expressions inside sum(), max(), min()")
print("     for memory-efficient aggregation of large datasets.")

List type:      <class 'list'>
Generator type: <class 'generator'>
List memory:    184 bytes
Generator mem:  224 bytes

vs_list[0] = 160.8
vs_list[3] = 240.4

Generator values (consumed one by one):
  160.8  186.4  211.7  240.4  256.0  276.2  296.2  313.7  331.3  350.8

Avg Vs = 262.3 m/s
Max Vs = 350.8 m/s
Min Vs = 160.8 m/s

Tip: Use generator expressions inside sum(), max(), min()
     for memory-efficient aggregation of large datasets.


---

## Part 10: `itertools` for Engineering -- `chain` and `islice`

The `itertools` module provides memory-efficient building blocks for working with iterators.

| Tool | What It Does | Geotechnical Use |
|------|-------------|------------------|
| `chain(*iterables)` | Concatenate multiple iterables | Combine layers from multiple boreholes into one stream |
| `islice(iterable, n)` | Take first N items | Sample the top N layers from a deep borehole |

In [11]:
# ─────────────────────────────────────────────────────
# itertools -- chain and islice for borehole data
# ─────────────────────────────────────────────────────

from itertools import chain, islice

# Three borehole datasets
bh01 = [
    {"bh": "BH-01", "depth": 1.5, "N60": 5,  "soil": "Fill"},
    {"bh": "BH-01", "depth": 3.0, "N60": 12, "soil": "Sand"},
    {"bh": "BH-01", "depth": 4.5, "N60": 22, "soil": "Sand"},
    {"bh": "BH-01", "depth": 6.0, "N60": 35, "soil": "Gravel"},
]

bh02 = [
    {"bh": "BH-02", "depth": 1.5, "N60": 3,  "soil": "Fill"},
    {"bh": "BH-02", "depth": 3.0, "N60": 8,  "soil": "Clay"},
    {"bh": "BH-02", "depth": 4.5, "N60": 18, "soil": "Sand"},
    {"bh": "BH-02", "depth": 6.0, "N60": 28, "soil": "Gravel"},
]

bh03 = [
    {"bh": "BH-03", "depth": 1.5, "N60": 7,  "soil": "Fill"},
    {"bh": "BH-03", "depth": 3.0, "N60": 15, "soil": "Sand"},
    {"bh": "BH-03", "depth": 4.5, "N60": 25, "soil": "Sand"},
    {"bh": "BH-03", "depth": 6.0, "N60": 40, "soil": "Gravel"},
]

# chain: combine all boreholes into one stream
print("All layers from all boreholes (chained):")
print(f"  {'BH':6s} {'Depth':>5s} {'N60':>4s} {'Soil':12s}")
print("  " + "─" * 30)
for layer in chain(bh01, bh02, bh03):
    print(f"  {layer['bh']:6s} {layer['depth']:4.1f}m {layer['N60']:4d} {layer['soil']:12s}")
print()

# islice: get only the first 5 layers from the combined stream
print("First 5 layers across all boreholes:")
combined = chain(bh01, bh02, bh03)
for layer in islice(combined, 5):
    print(f"  {layer['bh']} | {layer['depth']}m | N={layer['N60']} | {layer['soil']}")
print()

# islice: skip first 2, take next 4
print("Skip 2, take 4 (layers 3-6):")
combined = chain(bh01, bh02, bh03)
for layer in islice(combined, 2, 6):
    print(f"  {layer['bh']} | {layer['depth']}m | N={layer['N60']} | {layer['soil']}")

All layers from all boreholes (chained):
  BH     Depth  N60 Soil        
  ──────────────────────────────
  BH-01   1.5m    5 Fill        
  BH-01   3.0m   12 Sand        
  BH-01   4.5m   22 Sand        
  BH-01   6.0m   35 Gravel      
  BH-02   1.5m    3 Fill        
  BH-02   3.0m    8 Clay        
  BH-02   4.5m   18 Sand        
  BH-02   6.0m   28 Gravel      
  BH-03   1.5m    7 Fill        
  BH-03   3.0m   15 Sand        
  BH-03   4.5m   25 Sand        
  BH-03   6.0m   40 Gravel      

First 5 layers across all boreholes:
  BH-01 | 1.5m | N=5 | Fill
  BH-01 | 3.0m | N=12 | Sand
  BH-01 | 4.5m | N=22 | Sand
  BH-01 | 6.0m | N=35 | Gravel
  BH-02 | 1.5m | N=3 | Fill

Skip 2, take 4 (layers 3-6):
  BH-01 | 4.5m | N=22 | Sand
  BH-01 | 6.0m | N=35 | Gravel
  BH-02 | 1.5m | N=3 | Fill
  BH-02 | 3.0m | N=8 | Clay


---

## Part 11: Complete Application -- Monitored Geotechnical Analysis Pipeline

Now we combine everything: **decorators** for timing, logging, and validation, plus **generators** for memory-efficient streaming through a large multi-borehole dataset.

The pipeline:
1. **Generator** streams layers from multiple boreholes
2. **Generator** filters to competent bearing layers
3. **Decorated function** computes corrected bearing capacity
4. All calls are timed, validated, and logged automatically

In [12]:
# ══════════════════════════════════════════════════════
# COMPLETE APPLICATION: Monitored Geotechnical Pipeline
# Decorators + Generators working together
# ══════════════════════════════════════════════════════

import time
import math
import functools
from itertools import chain, islice
from datetime import datetime

# ── DECORATORS ──────────────────────────────────────

pipeline_log = []

def pipeline_timer(func):
    """Decorator: time each function call."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        pipeline_log.append({
            "time": datetime.now().strftime("%H:%M:%S"),
            "func": func.__name__,
            "elapsed_ms": round(elapsed * 1000, 3),
        })
        return result
    return wrapper

def require_positive(*param_names):
    """Decorator factory: validate that named params are positive."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            import inspect
            sig = inspect.signature(func)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            for name in param_names:
                val = bound.arguments.get(name)
                if val is not None and val <= 0:
                    raise ValueError(f"{func.__name__}: '{name}' must be > 0, got {val}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

# ── GENERATORS ──────────────────────────────────────

def stream_site_data(boreholes):
    """
    Generator: streams all layers from all boreholes,
    one layer at a time. Adds borehole ID and water table
    to each layer record.
    """
    for bh_id, bh_data in boreholes.items():
        wt = bh_data["wt"]
        for layer in bh_data["layers"]:
            yield {
                "borehole": bh_id,
                "wt": wt,
                **layer
            }

def filter_bearing_strata(layer_stream, min_n60=10):
    """
    Generator: passes through only layers suitable as
    bearing strata (N60 >= min_n60).
    """
    for layer in layer_stream:
        if layer["N60"] >= min_n60:
            yield layer

# ── DECORATED COMPUTATION ───────────────────────────

@pipeline_timer
@require_positive('n60', 'depth')
def compute_corrected_capacity(n60, depth, wt, gamma=18.0):
    """
    Compute corrected allowable bearing capacity.
    1. Effective stress at depth
    2. Overburden correction (Liao & Whitman)
    3. Meyerhof bearing capacity
    """
    # Effective stress
    gamma_w = 9.81
    if depth > wt:
        sigma_v_eff = gamma * wt + (gamma - gamma_w) * (depth - wt)
    else:
        sigma_v_eff = gamma * depth
    
    # Overburden correction
    cn = min(math.sqrt(100.0 / sigma_v_eff), 2.0) if sigma_v_eff > 0 else 1.0
    n1_60 = n60 * cn
    
    # Meyerhof (simplified)
    q_a = n1_60 / 8.0  # kPa (simplified for illustration)
    
    return round(q_a, 2)

# ── SITE DATA ──────────────────────────────────────

site_boreholes = {
    "BH-01": {
        "wt": 3.0,
        "layers": [
            {"depth": 1.5, "N60": 4,  "soil": "Fill"},
            {"depth": 3.0, "N60": 8,  "soil": "Silty Sand"},
            {"depth": 4.5, "N60": 15, "soil": "Sand"},
            {"depth": 6.0, "N60": 22, "soil": "Clayey Sand"},
            {"depth": 7.5, "N60": 30, "soil": "Dense Sand"},
            {"depth": 9.0, "N60": 42, "soil": "Gravel"},
        ]
    },
    "BH-02": {
        "wt": 2.5,
        "layers": [
            {"depth": 1.5, "N60": 3,  "soil": "Fill"},
            {"depth": 3.0, "N60": 6,  "soil": "Clay"},
            {"depth": 4.5, "N60": 12, "soil": "Sand"},
            {"depth": 6.0, "N60": 20, "soil": "Sand"},
            {"depth": 7.5, "N60": 28, "soil": "Dense Sand"},
            {"depth": 9.0, "N60": 38, "soil": "Gravel"},
        ]
    },
    "BH-03": {
        "wt": 3.5,
        "layers": [
            {"depth": 1.5, "N60": 6,  "soil": "Fill"},
            {"depth": 3.0, "N60": 10, "soil": "Silty Sand"},
            {"depth": 4.5, "N60": 18, "soil": "Sand"},
            {"depth": 6.0, "N60": 25, "soil": "Clayey Sand"},
            {"depth": 7.5, "N60": 35, "soil": "Dense Sand"},
            {"depth": 9.0, "N60": 50, "soil": "Weathered Rock"},
        ]
    },
}

# ── RUN THE PIPELINE ───────────────────────────────

print("=" * 72)
print("  MONITORED GEOTECHNICAL ANALYSIS PIPELINE")
print("  Decorators: @pipeline_timer, @require_positive")
print("  Generators: stream_site_data -> filter_bearing_strata")
print("=" * 72)

# Build the generator pipeline (nothing executes yet)
all_layers = stream_site_data(site_boreholes)
bearing_layers = filter_bearing_strata(all_layers, min_n60=10)

# Process the stream
results = []
print(f"\n  {'BH':6s} {'Depth':>5s} {'N60':>4s} {'Soil':14s} {'q_a (kPa)':>10s}")
print("  " + "─" * 44)

for layer in bearing_layers:
    q_a = compute_corrected_capacity(
        n60=layer["N60"],
        depth=layer["depth"],
        wt=layer["wt"]
    )
    results.append({**layer, "q_a": q_a})
    print(f"  {layer['borehole']:6s} {layer['depth']:4.1f}m {layer['N60']:4d} "
          f"{layer['soil']:14s} {q_a:10.2f}")

print("  " + "─" * 44)
print(f"  Layers processed: {len(results)}")

  MONITORED GEOTECHNICAL ANALYSIS PIPELINE
  Decorators: @pipeline_timer, @require_positive
  Generators: stream_site_data -> filter_bearing_strata

  BH     Depth  N60 Soil            q_a (kPa)
  ────────────────────────────────────────────
  BH-01   4.5m   15 Sand                 2.30
  BH-01   6.0m   22 Clayey Sand          3.10
  BH-01   7.5m   30 Dense Sand           3.93
  BH-01   9.0m   42 Gravel               5.17
  BH-02   4.5m   12 Sand                 1.91
  BH-02   6.0m   20 Sand                 2.91
  BH-02   7.5m   28 Dense Sand           3.78
  BH-02   9.0m   38 Gravel               4.79
  BH-03   3.0m   10 Silty Sand           1.70
  BH-03   4.5m   18 Sand                 2.67
  BH-03   6.0m   25 Clayey Sand          3.42
  BH-03   7.5m   35 Dense Sand           4.47
  BH-03   9.0m   50 Weathered Rock       6.01
  ────────────────────────────────────────────
  Layers processed: 13


In [13]:
# ─────────────────────────────────────────────────────
# PIPELINE SUMMARY -- Review the audit trail
# ─────────────────────────────────────────────────────

# Summary statistics from results
all_qa = [r["q_a"] for r in results]
print("BEARING CAPACITY SUMMARY:")
print(f"  Min q_a: {min(all_qa):.2f} kPa")
print(f"  Max q_a: {max(all_qa):.2f} kPa")
print(f"  Avg q_a: {sum(all_qa)/len(all_qa):.2f} kPa")
print()

# Per-borehole summary
print("Per-borehole averages:")
bh_ids = sorted(set(r["borehole"] for r in results))
for bh in bh_ids:
    bh_qa = [r["q_a"] for r in results if r["borehole"] == bh]
    avg = sum(bh_qa) / len(bh_qa)
    print(f"  {bh}: {len(bh_qa)} bearing layers, avg q_a = {avg:.2f} kPa")
print()

# Pipeline performance log
print("PIPELINE PERFORMANCE LOG:")
print("─" * 55)
total_ms = sum(e["elapsed_ms"] for e in pipeline_log)
print(f"  Total computations:  {len(pipeline_log)}")
print(f"  Total time:          {total_ms:.3f} ms")
print(f"  Avg per computation: {total_ms/len(pipeline_log):.3f} ms")
print("─" * 55)
print()
print("Key insight: Generators streamed layers one-at-a-time,")
print("while decorators added timing and validation automatically.")
print("Neither required changes to the core computation function.")

BEARING CAPACITY SUMMARY:
  Min q_a: 1.70 kPa
  Max q_a: 6.01 kPa
  Avg q_a: 3.55 kPa

Per-borehole averages:
  BH-01: 4 bearing layers, avg q_a = 3.62 kPa
  BH-02: 4 bearing layers, avg q_a = 3.35 kPa
  BH-03: 5 bearing layers, avg q_a = 3.65 kPa

PIPELINE PERFORMANCE LOG:
───────────────────────────────────────────────────────
  Total computations:  13
  Total time:          0.609 ms
  Avg per computation: 0.047 ms
───────────────────────────────────────────────────────

Key insight: Generators streamed layers one-at-a-time,
while decorators added timing and validation automatically.
Neither required changes to the core computation function.


---

## Practice Exercises

### Exercise 1: Closure -- Unit Converter Factory
Write a `make_unit_converter(factor, unit_name)` closure that returns a function to convert values. Create converters for kPa-to-psf (factor=20.89) and m-to-ft (factor=3.281). Test each on a list of values.

### Exercise 2: Decorator -- `@count_calls`
Write a decorator that counts how many times a function is called. Store the count as `func.call_count`. Apply it to a `compute_vs(n60)` function, call it 10 times, and print the count.

### Exercise 3: Generator -- Borehole Log Reader
Write a generator `read_borehole(layers)` that yields each layer with an added `"layer_num"` field. Then write a second generator `flag_weak_layers(stream, threshold=10)` that adds a `"flag": "WEAK"` to layers with N60 below the threshold. Chain them together and process the output.

In [14]:
# EXERCISE 1
# def make_unit_converter(factor, unit_name):
#     def convert(value):
#         ...
#     return convert
#
# kpa_to_psf = make_unit_converter(20.89, "psf")
# m_to_ft = make_unit_converter(3.281, "ft")
# pressures = [50, 100, 150, 200]
# print([kpa_to_psf(p) for p in pressures])

In [15]:
# EXERCISE 2
# def count_calls(func):
#     @functools.wraps(func)
#     def wrapper(*args, **kwargs):
#         wrapper.call_count += 1
#         ...
#     wrapper.call_count = 0
#     return wrapper
#
# @count_calls
# def compute_vs(n60):
#     return round(97.0 * (n60 ** 0.314), 1)
#
# for n in [5, 8, 12, 18, 22, 28, 35, 42, 50, 60]:
#     compute_vs(n)
# print(f"Called {compute_vs.call_count} times")

In [16]:
# EXERCISE 3
# def read_borehole(layers):
#     for i, layer in enumerate(layers, 1):
#         yield {"layer_num": i, **layer}
#
# def flag_weak_layers(stream, threshold=10):
#     for layer in stream:
#         if layer["N60"] < threshold:
#             yield {**layer, "flag": "WEAK"}
#         else:
#             yield layer
#
# layers = [
#     {"depth": 1.5, "N60": 4, "soil": "Fill"},
#     {"depth": 3.0, "N60": 8, "soil": "Sand"},
#     {"depth": 4.5, "N60": 15, "soil": "Sand"},
#     {"depth": 6.0, "N60": 25, "soil": "Gravel"},
# ]
# for layer in flag_weak_layers(read_borehole(layers)):
#     print(layer)

---

## Key Takeaways

1. **Functions are first-class objects** in Python -- you can pass them as arguments, store them in lists, and return them from other functions. This lets you write generic calculators that accept different formulas.

2. **Closures** create customized functions that remember their creation context. The SPT hammer correction factory produces a unique corrector for each hammer type -- no global variables needed.

3. **Decorators** add behavior (timing, validation, logging) to any function with a single `@` line. Write the boilerplate once, apply it everywhere -- no copy-pasting validation checks.

4. **Stacking decorators** lets you compose multiple concerns: `@timer @log @validate` turns a simple function into a fully monitored, validated computation.

5. **Generators with `yield`** produce data on demand, one item at a time. For large borehole datasets, this means constant memory usage regardless of dataset size.

6. **Generator expressions** (`(x for x in data)`) are lazy list comprehensions -- use them inside `sum()`, `max()`, `min()` to avoid building intermediate lists.

7. **`itertools.chain`** and **`islice`** let you combine and sample from multiple data streams without loading everything into memory.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| First-class functions | Passing loss functions, activation functions to models |
| Closures | Learning rate schedulers, callback factories |
| `@decorator` | `@tf.function`, `@torch.no_grad()`, Flask `@app.route` |
| `@timer` | Profiling training loops, benchmarking inference |
| `@validate_input` | Input validation in ML pipelines, schema checks |
| `@log_calculation` | Experiment tracking (MLflow, Weights & Biases) |
| Generators / `yield` | `tf.data.Dataset`, PyTorch `DataLoader`, batch streaming |
| Generator expressions | Memory-efficient feature computation |
| `itertools.chain` | Combining multiple data sources for training |
| `itertools.islice` | Taking N samples for validation split |

In Phase 2, when we build ML pipelines, decorators will manage experiment logging and generators will stream training data in batches -- exactly the patterns we practiced today.

---

*Day 013 of 100 -- Decorators add superpowers to functions; generators make data flow efficiently!*